In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import torch.nn as nn
from torchinfo import summary
import statistics
import csv

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
all = []

with open('embedding_dataset.csv', 'r') as csvfile:
    reader = csv.reader(csvfile)
    next(reader) #skips first line of csv with headers
    for line in reader:
        all.append(line)
        # match line[0]:
        #     case 'Pinaceae':
        #         pin.append(line)
        #         all.append(line)
        #     case 'Betulaceae':
        #         bet.append(line)
        #         all.append(line)
        #     case 'Cupressaceae':
        #         cup.append(line)
        #         all.append(line)
        #     case 'Sapindaceae':
        #         sap.append(line)
        #         all.append(line)
        #     case 'Fagaceae':
        #         fag.append(line)
        #         all.append(line)
        #     case _:
        #         print('unplanned classification')

In [ ]:
print(len(all))

49627


In [ ]:
genus_toint_dict = {}
species_toint_dict = {}
family_toint_dict = {}
genus_tostring_dict = {}
species_tostring_dict = {}
family_tostring_dict = {}

fencode = 0
gencode = 0
sencode = 0

family = []
genus = []
species = []
year = []
embeddings = []

master = []

for entry in all:
    if not entry[0] in family_toint_dict:
        family_toint_dict[entry[0]] = fencode
        family_tostring_dict[fencode] = entry[0]
        fencode += 1
    if not entry[1] in genus_toint_dict:
        genus_toint_dict[entry[1]] = gencode
        genus_tostring_dict[gencode] = entry[1]
        gencode += 1
    if not entry[2] in species_toint_dict:
        species_toint_dict[entry[2]] = sencode
        species_tostring_dict[sencode] = entry[2]
        sencode += 1

for entry in all:
    family.append(family_toint_dict[entry[0]])
    genus.append(genus_toint_dict[entry[1]])
    species.append(species_toint_dict[entry[2]])
    year.append(entry[3])
    embeddings.append(entry[5:])

master = [family, genus, species, year, embeddings]

# print(master[2])
# print(genus_toint_dict)
# print(species_toint_dict)
# print(genus_tostring_dict)
# print(species_tostring_dict)

In [ ]:
# pin = []
# bet = []
# cup = []
# sap = []
# fag = []

print(len(master[0]))

49627


In [ ]:
class embed_dataset(torch.utils.data.Dataset):
    def __init__(self, family, genus, species, year, embeddings):
        self.family = family
        self.genus = genus
        self.species = species
        self.year = year
        self.embeddings = embeddings

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        family = self.family[idx]
        genus = self.genus[idx]
        species = self.species[idx]
        year = self.year[idx]
        embedding = self.embeddings[idx]
        return family, genus, species, year, embedding


# Create dataset splits
# Set up dataloader for batches
batch_size = 1

train_dataset = embed_dataset(family, genus, species, year, embeddings)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)